# 05 — Model Training

## Smart India Real Estate Analytics

### Objective

The objective of this notebook is to train and compare multiple regression models for real-estate property price prediction.

The models will be trained using the processed training data prepared in `04_preprocessing.ipynb`.

### Training Strategy

The following stages will be followed:

1. Load the prepared training and test data.
2. Establish a simple baseline model.
3. Train multiple suitable regression algorithms.
4. Evaluate models using consistent metrics.
5. Compare model performance.
6. Identify the most promising model for further tuning.
7. Tune the selected model only if the baseline results justify it.

### Evaluation Metrics

The primary regression metrics will be:

- **MAE (Mean Absolute Error)** — average absolute prediction error.
- **RMSE (Root Mean Squared Error)** — penalizes larger prediction errors more strongly.
- **R² (Coefficient of Determination)** — measures the proportion of target variance explained by the model.

### Important Principles

- The test set will remain unseen during model selection and tuning.
- The same processed training and test datasets will be used for fair model comparison.
- No target-derived feature will be introduced.
- Model training will initially be performed locally.
- Computational requirements will be monitored before considering cloud training.
- Random states will be fixed wherever applicable for reproducibility.

## Load Preprocessing Artifact

The fitted preprocessing transformer created during `04_preprocessing.ipynb` is loaded from the `models/` directory.

The saved transformer will be reused for model development so that the same preprocessing configuration is maintained across training, evaluation, and future deployment.

The transformer will not be refitted in this notebook.

In [1]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../models")
PREPROCESSOR_PATH = MODEL_DIR / "preprocessor.joblib"

if not PREPROCESSOR_PATH.exists():
    raise FileNotFoundError(
        f"Preprocessor not found: {PREPROCESSOR_PATH}"
    )

preprocessor = joblib.load(PREPROCESSOR_PATH)

print("Preprocessor loaded successfully.")

Preprocessor loaded successfully.


## Load Processed Training and Test Data

The processed training and test datasets were generated and validated in `04_preprocessing.ipynb` and saved as reusable artifacts.

This notebook loads those artifacts directly rather than repeating the preprocessing process.

The preprocessing transformer is already saved separately and will be reused consistently.

In [3]:
from scipy import sparse
import pandas as pd

X_train_processed = sparse.load_npz(
    MODEL_DIR / "X_train_processed.npz"
)

X_test_processed = sparse.load_npz(
    MODEL_DIR / "X_test_processed.npz"
)

y_train = pd.read_csv(
    MODEL_DIR / "y_train.csv"
)["Price"]

y_test = pd.read_csv(
    MODEL_DIR / "y_test.csv"
)["Price"]

print("Training features:", X_train_processed.shape)
print("Training target:", y_train.shape)

print("Test features:", X_test_processed.shape)
print("Test target:", y_test.shape)

Training features: (11620, 1738)
Training target: (11620,)
Test features: (2905, 1738)
Test target: (2905,)


## Validate Training Data

The processed feature matrices and target vectors are loaded from the saved artifacts created during preprocessing.

Before model training, their dimensions and target alignment are verified to ensure that the correct datasets are being used.

In [4]:
print("Training features:", X_train_processed.shape)
print("Training target:", y_train.shape)

print("Test features:", X_test_processed.shape)
print("Test target:", y_test.shape)

print(
    "\nTraining rows aligned:",
    X_train_processed.shape[0] == len(y_train)
)

print(
    "Test rows aligned:",
    X_test_processed.shape[0] == len(y_test)
)

print(
    "\nTraining matrix type:",
    type(X_train_processed).__name__
)

print(
    "Test matrix type:",
    type(X_test_processed).__name__
)

Training features: (11620, 1738)
Training target: (11620,)
Test features: (2905, 1738)
Test target: (2905,)

Training rows aligned: True
Test rows aligned: True

Training matrix type: csr_matrix
Test matrix type: csr_matrix


## Baseline Model — Linear Regression

Linear Regression is used as the initial baseline model.

It assumes that the target price can be approximated as a linear combination of the processed input features.

The purpose of this model is not necessarily to achieve the best prediction performance. Instead, it provides a reference point against which more complex nonlinear models can be compared.

The model will be trained using the processed training data and evaluated on the held-out test data.

In [5]:
from sklearn.linear_model import LinearRegression

baseline_model = LinearRegression()

baseline_model.fit(
    X_train_processed,
    y_train
)

print("Linear Regression trained successfully.")

Linear Regression trained successfully.


## Baseline Predictions

The trained Linear Regression model will now generate predictions for the held-out test set.

The test features are used only for prediction and evaluation; they are not used during model fitting.

In [6]:
baseline_predictions = baseline_model.predict(X_test_processed)

print("Baseline predictions generated successfully.")
print("Number of predictions:", len(baseline_predictions))

Baseline predictions generated successfully.
Number of predictions: 2905


## Model Evaluation Metrics

Regression models will be evaluated using three metrics:

- **MAE (Mean Absolute Error):** average absolute difference between actual and predicted prices.
- **RMSE (Root Mean Squared Error):** gives greater weight to large prediction errors.
- **R² Score:** measures how much of the variation in property prices is explained by the model.

The same evaluation function will be used for all candidate models to ensure a fair comparison.

In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate_regression_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

In [8]:
baseline_metrics = evaluate_regression_model(
    y_test,
    baseline_predictions
)

baseline_metrics

{'MAE': 6187993.960731125,
 'RMSE': np.float64(20831562.31475336),
 'R2': 0.1124685773657722}

## Naive Baseline — Median Price

A simple naive baseline is created by predicting the median training-set property price for every test observation.

This model does not use any input features. It provides a minimum reference point for evaluating whether the machine-learning models provide meaningful predictive improvement.

The median is calculated from the training target only.

In [9]:
from sklearn.dummy import DummyRegressor

dummy_model = DummyRegressor(strategy="median")

dummy_model.fit(
    X_train_processed,
    y_train
)

dummy_predictions = dummy_model.predict(X_test_processed)

dummy_metrics = evaluate_regression_model(
    y_test,
    dummy_predictions
)

dummy_metrics

{'MAE': 6996584.509466438,
 'RMSE': np.float64(22538765.923696153),
 'R2': -0.0389635706941478}

## Baseline Model Comparison

Two baseline approaches were evaluated:

1. **Median Dummy Regressor** — predicts the median training price for every property.
2. **Linear Regression** — provides a simple feature-based predictive baseline.

Linear Regression performs better than the naive median baseline, but its predictive performance remains limited. More flexible nonlinear models will therefore be evaluated next.

In [10]:
baseline_results = pd.DataFrame([
    {
        "Model": "Median Dummy",
        **dummy_metrics
    },
    {
        "Model": "Linear Regression",
        **baseline_metrics
    }
])

baseline_results

,Model,MAE,RMSE,R2
0,Median Dummy,6.996585e+06,2.253877e+07,-0.038964
1,Linear Regression,6.187994e+06,2.083156e+07,0.112469


## Nonlinear Baseline — Random Forest

Random Forest Regressor is introduced as the first nonlinear model.

Unlike Linear Regression, Random Forest can capture nonlinear relationships and interactions between property characteristics.

Because the processed dataset contains 1,738 features and 11,620 training observations, an initial conservative configuration will be used to control training time and memory usage.

The initial model is intended for comparison rather than final hyperparameter optimization.

In [11]:
from sklearn.ensemble import RandomForestRegressor

random_forest_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(
    X_train_processed,
    y_train
)

print("Random Forest trained successfully.")

Random Forest trained successfully.


## Random Forest Predictions

The trained Random Forest model will now generate predictions for the held-out test set.

The test data is used only for prediction and evaluation and has not been used during model training.

In [12]:
random_forest_predictions = random_forest_model.predict(
    X_test_processed
)

print("Random Forest predictions generated successfully.")
print("Number of predictions:", len(random_forest_predictions))

Random Forest predictions generated successfully.
Number of predictions: 2905


## Random Forest Evaluation

The Random Forest predictions will be evaluated using the same MAE, RMSE, and R² metrics used for the baseline models.

Using identical evaluation metrics allows a fair comparison between Linear Regression, the naive baseline, and Random Forest.

In [13]:
random_forest_metrics = evaluate_regression_model(
    y_test,
    random_forest_predictions
)

random_forest_metrics

{'MAE': 5428821.559840589,
 'RMSE': np.float64(20282974.724542193),
 'R2': 0.1585983630671084}

## Nonlinear Model — Gradient Boosting

Gradient Boosting is evaluated as another nonlinear regression approach.

Unlike Random Forest, which builds many independent decision trees and averages their predictions, Gradient Boosting builds trees sequentially, with each new tree attempting to correct errors made by the previous trees.

This approach can capture complex relationships between property characteristics and price.

An initial conservative configuration will be used for baseline comparison before any hyperparameter tuning.

In [14]:
from sklearn.ensemble import HistGradientBoostingRegressor

# Convert sparse matrices to dense float32 for HistGradientBoosting
X_train_hgb = X_train_processed.toarray().astype(np.float32)
X_test_hgb = X_test_processed.toarray().astype(np.float32)

print("Dense training shape:", X_train_hgb.shape)
print("Dense test shape:", X_test_hgb.shape)

hist_gradient_model = HistGradientBoostingRegressor(
    max_iter=200,
    learning_rate=0.08,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

hist_gradient_model.fit(
    X_train_hgb,
    y_train
)

print("HistGradientBoosting trained successfully.")

Dense training shape: (11620, 1738)
Dense test shape: (2905, 1738)
HistGradientBoosting trained successfully.


## HistGradientBoosting Predictions

The trained HistGradientBoosting model will now generate predictions for the held-out test set.

The same test set used for the previous models will be used so that the model comparison remains fair.

In [15]:
hist_gradient_predictions = hist_gradient_model.predict(
    X_test_hgb
)

print(
    "HistGradientBoosting predictions generated successfully."
)

print(
    "Number of predictions:",
    len(hist_gradient_predictions)
)

HistGradientBoosting predictions generated successfully.
Number of predictions: 2905


## HistGradientBoosting Evaluation

The HistGradientBoosting predictions are evaluated using MAE, RMSE, and R².

The results will be compared with the naive baseline, Linear Regression, and Random Forest using the same test set and evaluation procedure.

In [16]:
hist_gradient_metrics = evaluate_regression_model(
    y_test,
    hist_gradient_predictions
)

hist_gradient_metrics

{'MAE': 5242686.951036572,
 'RMSE': np.float64(19692966.970228106),
 'R2': 0.20683716090378168}

## Initial Model Comparison

The initial model comparison shows how predictive performance changes as model complexity increases.

The naive median model provides a reference point, while Linear Regression establishes a simple feature-based baseline. Random Forest and HistGradientBoosting introduce nonlinear modeling capabilities.

The best-performing model at this stage will be selected for further investigation rather than immediately declared as the final production model.

In [17]:
model_results = pd.DataFrame([
    {
        "Model": "Median Dummy",
        **dummy_metrics
    },
    {
        "Model": "Linear Regression",
        **baseline_metrics
    },
    {
        "Model": "Random Forest",
        **random_forest_metrics
    },
    {
        "Model": "HistGradientBoosting",
        **hist_gradient_metrics
    }
])

model_results = model_results.sort_values(
    "RMSE"
).reset_index(drop=True)

model_results

,Model,MAE,RMSE,R2
0,HistGradientBoosting,5.242687e+06,1.969297e+07,0.206837
1,Random Forest,5.428822e+06,2.028297e+07,0.158598
2,Linear Regression,6.187994e+06,2.083156e+07,0.112469
3,Median Dummy,6.996585e+06,2.253877e+07,-0.038964


## Generalization Check

The current best-performing model, HistGradientBoosting, will be evaluated on both the training and test datasets.

Comparing training and test performance helps identify potential overfitting before hyperparameter tuning.

A large difference between training and test performance may indicate that the model has learned patterns specific to the training data rather than generalizable relationships.

In [19]:
hist_train_predictions = hist_gradient_model.predict(
    X_train_hgb
)

hist_train_metrics = evaluate_regression_model(
    y_train,
    hist_train_predictions
)

hist_test_metrics = evaluate_regression_model(
    y_test,
    hist_gradient_predictions
)

generalization_results = pd.DataFrame([
    {
        "Dataset": "Training",
        **hist_train_metrics
    },
    {
        "Dataset": "Test",
        **hist_test_metrics
    }
])

generalization_results

,Dataset,MAE,RMSE,R2
0,Training,4.706455e+06,1.349704e+07,0.419193
1,Test,5.242687e+06,1.969297e+07,0.206837


## HistGradientBoosting Hyperparameter Tuning

HistGradientBoosting showed the best initial test performance but also displayed a generalization gap between training and test performance.

A controlled hyperparameter search will therefore be performed to determine whether model complexity can be adjusted to improve test performance and reduce overfitting.

The search space is intentionally limited to keep computational cost reasonable.

The test set will remain untouched during tuning.

## Hyperparameter Search

A controlled randomized hyperparameter search will be used to improve the HistGradientBoosting model.

Three-fold cross-validation will be performed using the training data only. The test set will remain completely untouched during hyperparameter selection.

The search will focus on parameters controlling model complexity, learning rate, number of iterations, and regularization.

The selected configuration will then be retrained on the complete training set and evaluated on the held-out test set.

In [20]:
from sklearn.model_selection import RandomizedSearchCV

hgb_param_grid = {
    "max_iter": [150, 200, 250],
    "learning_rate": [0.05, 0.08, 0.1],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [20, 30, 50],
    "l2_regularization": [0.0, 1.0, 5.0, 10.0]
}

hgb_tuning_model = HistGradientBoostingRegressor(
    random_state=42
)

hgb_random_search = RandomizedSearchCV(
    estimator=hgb_tuning_model,
    param_distributions=hgb_param_grid,
    n_iter=6,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=42,
    n_jobs=2,
    verbose=1,
    return_train_score=True
)

print("Hyperparameter search configured successfully.")
print("Candidates:", 6)
print("Cross-validation folds:", 3)
print("Total model fits:", 6 * 3)

Hyperparameter search configured successfully.
Candidates: 6
Cross-validation folds: 3
Total model fits: 18


In [21]:
import time

print("Starting HistGradientBoosting hyperparameter search...")

start_time = time.time()

hgb_random_search.fit(
    X_train_hgb,
    y_train
)

elapsed_time = time.time() - start_time

print("\nHyperparameter search completed.")
print(f"Search time: {elapsed_time:.2f} seconds")
print("Best parameters:")
print(hgb_random_search.best_params_)
print(f"Best CV RMSE: {-hgb_random_search.best_score_:.2f}")

Starting HistGradientBoosting hyperparameter search...
Fitting 3 folds for each of 6 candidates, totalling 18 fits

Hyperparameter search completed.
Search time: 516.81 seconds
Best parameters:
{'min_samples_leaf': 50, 'max_leaf_nodes': 15, 'max_iter': 150, 'learning_rate': 0.1, 'l2_regularization': 1.0}
Best CV RMSE: 14441509.50


## Tuned HistGradientBoosting Evaluation

The best HistGradientBoosting configuration identified through 3-fold cross-validation is already refitted on the complete training dataset by `RandomizedSearchCV`.

The tuned model will now be evaluated on the untouched test set.

The test set has not been used during hyperparameter selection and therefore provides an unbiased estimate of the tuned model's generalization performance.

In [22]:
tuned_hgb_model = hgb_random_search.best_estimator_

tuned_hgb_predictions = tuned_hgb_model.predict(
    X_test_hgb
)

tuned_hgb_metrics = evaluate_regression_model(
    y_test,
    tuned_hgb_predictions
)

print("Tuned HistGradientBoosting test results:")
print(tuned_hgb_metrics)

Tuned HistGradientBoosting test results:
{'MAE': 5181651.456001289, 'RMSE': np.float64(19559862.63255491), 'R2': 0.2175228671574112}


In [23]:
hgb_comparison = pd.DataFrame([
    {
        "Model": "Original HistGradientBoosting",
        **hist_gradient_metrics
    },
    {
        "Model": "Tuned HistGradientBoosting",
        **tuned_hgb_metrics
    }
])

hgb_comparison

,Model,MAE,RMSE,R2
0,Original HistGradientBoosting,5.242687e+06,1.969297e+07,0.206837
1,Tuned HistGradientBoosting,5.181651e+06,1.955986e+07,0.217523


## Tuned Model Selection

The tuned HistGradientBoosting model outperformed the original configuration on the untouched test set.

The tuned model achieved:

- MAE: approximately ₹51.82 lakh
- RMSE: approximately ₹1.96 crore
- R²: approximately 0.218

The improvement was consistent across MAE, RMSE, and R². Therefore, the tuned HistGradientBoosting model is selected as the current best candidate for further error analysis.

Further model development will focus on understanding prediction errors rather than introducing target-derived features solely to increase performance.

## Prediction Error Analysis

The selected tuned HistGradientBoosting model will be analyzed using its individual test-set predictions.

Prediction errors will be calculated as the difference between actual and predicted prices.

This analysis will help identify:

- the distribution of prediction errors
- unusually large errors
- systematic underprediction or overprediction
- whether the model performs consistently across the test dataset

The analysis uses the untouched test predictions generated by the selected model.

In [24]:
error_analysis = pd.DataFrame({
    "Actual_Price": y_test.to_numpy(),
    "Predicted_Price": tuned_hgb_predictions
})

error_analysis["Error"] = (
    error_analysis["Actual_Price"]
    - error_analysis["Predicted_Price"]
)

error_analysis["Absolute_Error"] = (
    error_analysis["Error"].abs()
)

error_analysis["Percentage_Error"] = (
    error_analysis["Absolute_Error"]
    / error_analysis["Actual_Price"].replace(0, np.nan)
) * 100

print("Error analysis dataset shape:", error_analysis.shape)

error_analysis.head()

Error analysis dataset shape: (2905, 5)


,Actual_Price,Predicted_Price,Error,Absolute_Error,Percentage_Error
0,19000000.0,1.320831e+07,5.791693e+06,5.791693e+06,30.482595
1,15000000.0,7.525913e+06,7.474087e+06,7.474087e+06,49.827247
2,12500000.0,2.604224e+07,-1.354224e+07,1.354224e+07,108.337900
3,4500000.0,5.796019e+06,-1.296019e+06,1.296019e+06,28.800428
4,8600000.0,1.176261e+07,-3.162611e+06,3.162611e+06,36.774549


## Largest Prediction Errors

The test predictions will be sorted by absolute prediction error to identify properties for which the selected model performs poorly.

Examining these cases can reveal whether large errors are associated with unusually expensive properties, unusual property configurations, extreme areas, or limitations in the available features.

In [25]:
largest_errors = (
    error_analysis
    .sort_values("Absolute_Error", ascending=False)
    .head(20)
)

largest_errors

,Actual_Price,Predicted_Price,Error,Absolute_Error,Percentage_Error
1561,840000000.0,7.936609e+06,8.320634e+08,8.320634e+08,99.055166
367,250000000.0,4.047120e+07,2.095288e+08,2.095288e+08,83.811519
2726,260000000.0,8.413181e+07,1.758682e+08,1.758682e+08,67.641610
223,185000000.0,1.821873e+07,1.667813e+08,1.667813e+08,90.152038
1522,210000000.0,7.258375e+07,1.374162e+08,1.374162e+08,65.436310
2421,150000000.0,1.554145e+07,1.344586e+08,1.344586e+08,89.639035
1488,160000000.0,3.891799e+07,1.210820e+08,1.210820e+08,75.676256
1383,150000000.0,3.407157e+07,1.159284e+08,1.159284e+08,77.285621
1535,120000000.0,5.417572e+06,1.145824e+08,1.145824e+08,95.485356
2841,200000000.0,8.900610e+07,1.109939e+08,1.109939e+08,55.496950


## Error Analysis by Price Range

The model's prediction performance will be analyzed across different actual-price ranges.

This helps determine whether prediction errors are concentrated in particular segments of the real-estate market.

The analysis will use absolute error and mean absolute error rather than relying only on percentage error, because percentage error can become disproportionately large for low-priced properties.

In [26]:
error_by_price = error_analysis.copy()

error_by_price["Price_Range"] = pd.cut(
    error_by_price["Actual_Price"],
    bins=[
        0,
        50_00_000,
        1_00_00_000,
        2_00_00_000,
        5_00_00_000,
        10_00_00_000,
        np.inf
    ],
    labels=[
        "< ₹50L",
        "₹50L–₹1Cr",
        "₹1Cr–₹2Cr",
        "₹2Cr–₹5Cr",
        "₹5Cr–₹10Cr",
        "> ₹10Cr"
    ],
    include_lowest=True
)

price_range_analysis = (
    error_by_price
    .groupby("Price_Range", observed=False)
    .agg(
        Properties=("Actual_Price", "count"),
        MAE=("Absolute_Error", "mean"),
        Mean_Actual_Price=("Actual_Price", "mean")
    )
    .reset_index()
)

price_range_analysis

,Price_Range,Properties,MAE,Mean_Actual_Price
0,< ₹50L,1092,2.295008e+06,3.229329e+06
1,₹50L–₹1Cr,990,2.824365e+06,7.259466e+06
2,₹1Cr–₹2Cr,539,5.749713e+06,1.421447e+07
3,₹2Cr–₹5Cr,222,1.149488e+07,3.002613e+07
4,₹5Cr–₹10Cr,46,3.410056e+07,7.083478e+07
5,> ₹10Cr,16,1.581777e+08,2.039375e+08


## Error Analysis Findings

The error analysis shows that prediction error increases substantially with property price.

The model performs comparatively better on lower-priced properties and has much larger absolute errors for high-value properties. This is partly influenced by the strongly right-skewed distribution of property prices.

The highest-price segment contains relatively few observations, so conclusions about this segment should be interpreted cautiously.

The findings motivate testing a log-transformed target as an additional modeling experiment. The original price target and dataset will remain unchanged.

## Log-Target HistGradientBoosting

Because property prices are strongly right-skewed, a log-transformed target will be tested as an additional modeling approach.

The transformation used is:

`log1p(Price)`

The model will learn the transformed target, and predictions will be converted back to the original rupee scale using:

`expm1(prediction)`

The final MAE, RMSE, and R² will be calculated on the original price scale so that the results remain directly comparable with the other models.

The original `Price` values remain unchanged.

In [27]:
# Transform the training target only
y_train_log = np.log1p(y_train)

print("Original target:")
print(y_train.describe())

print("\nLog-transformed target:")
print(y_train_log.describe())

Original target:
count    1.162000e+04
mean     1.062712e+07
std      1.771092e+07
min      5.500000e+04
25%      3.700000e+06
50%      6.500000e+06
75%      1.120000e+07
max      6.500000e+08
Name: Price, dtype: float64

Log-transformed target:
count    11620.000000
mean        15.713733
std          0.896667
min         10.915107
25%         15.123844
50%         15.687313
75%         16.231424
max         20.292483
Name: Price, dtype: float64


In [28]:
log_hgb_model = HistGradientBoostingRegressor(
    max_iter=150,
    learning_rate=0.1,
    max_leaf_nodes=15,
    min_samples_leaf=50,
    l2_regularization=1.0,
    random_state=42
)

log_hgb_model.fit(
    X_train_hgb,
    y_train_log
)

print("Log-target HistGradientBoosting trained successfully.")

Log-target HistGradientBoosting trained successfully.


## Log-Target Model Evaluation

The log-target model predicts `log1p(Price)`. These predictions will be converted back to the original price scale using the inverse `expm1` transformation.

MAE, RMSE, and R² will then be calculated using the original price values so that this model can be fairly compared with the existing HistGradientBoosting model.

In [29]:
log_hgb_log_predictions = log_hgb_model.predict(
    X_test_hgb
)

# Convert predictions back to original Price scale
log_hgb_predictions = np.expm1(
    log_hgb_log_predictions
)

# Prevent any tiny numerical negative values
log_hgb_predictions = np.maximum(
    log_hgb_predictions,
    0
)

log_hgb_metrics = evaluate_regression_model(
    y_test,
    log_hgb_predictions
)

print("Log-target HistGradientBoosting results:")
print(log_hgb_metrics)

Log-target HistGradientBoosting results:
{'MAE': 4862765.155794755, 'RMSE': np.float64(19876327.94142232), 'R2': 0.19199813934259646}


In [30]:
log_model_comparison = pd.DataFrame([
    {
        "Model": "Tuned HistGradientBoosting",
        **tuned_hgb_metrics
    },
    {
        "Model": "Log-Target HistGradientBoosting",
        **log_hgb_metrics
    }
])

log_model_comparison

,Model,MAE,RMSE,R2
0,Tuned HistGradientBoosting,5.181651e+06,1.955986e+07,0.217523
1,Log-Target HistGradientBoosting,4.862765e+06,1.987633e+07,0.191998


## Log-Target Experiment Result

A log-transformed target was evaluated using the same tuned HistGradientBoosting configuration.

The log-target model achieved a lower MAE than the original-target model, indicating improved typical absolute error. However, its RMSE and R² were worse.

Since the project evaluates overall predictive performance using MAE, RMSE, and R² together, the original-target tuned HistGradientBoosting model remains the primary model.

The log-target approach is retained as an experimental comparison rather than selected as the final model.

## Save Selected Model

The tuned HistGradientBoosting model trained on the original price target is selected as the current primary model.

The trained model is saved as a reusable artifact so that the application can load the model without retraining it.

The preprocessing transformer remains stored separately and must be applied to new input data before prediction.

In [31]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_MODEL_PATH = MODEL_DIR / "final_model.joblib"

joblib.dump(
    tuned_hgb_model,
    FINAL_MODEL_PATH
)

print(f"Final model saved to: {FINAL_MODEL_PATH}")

Final model saved to: ..\models\final_model.joblib


## Final Model Training Summary

The model-training stage evaluated multiple regression approaches using the same held-out test set.

### Models evaluated

1. Median Dummy Regressor
2. Linear Regression
3. Random Forest Regressor
4. HistGradientBoosting Regressor
5. Tuned HistGradientBoosting Regressor
6. Log-Target HistGradientBoosting Regressor

### Selected Model

The tuned HistGradientBoosting model trained on the original `Price` target was selected as the primary model.

### Final Test Performance

- MAE: approximately ₹51.82 lakh
- RMSE: approximately ₹1.96 crore
- R²: approximately 0.218

The selected model was saved as `final_model.joblib` in the `models/` directory.

The preprocessing transformer is stored separately as `preprocessor.joblib` and must be reused when making predictions on new property data.

The test set remained untouched during hyperparameter selection and was used only for final model evaluation.